In [1]:
import kagglehub, os, glob, hashlib
IMG  = kagglehub.dataset_download("ashwanisingh997/ham10000-images-1229692")
SPL  = kagglehub.dataset_download("ashwanisingh997/ham10000-splits-1229692")
CODE = kagglehub.dataset_download("ashwanisingh997/thesis-code-1229692")
CA   = kagglehub.dataset_download("ashwanisingh997/run1-curve-a-1229692")
CB   = kagglehub.dataset_download("ashwanisingh997/run1-curve-b-1229692")

h = hashlib.sha256(open(f"{CODE}/metrics.py", "rb").read()).hexdigest().upper()
assert h == "602747135A78568EAF96E64DB70A7C43CAC6F8527089DBE26711B0D834F05E17", "metrics.py is not v4/v5"
print("code: metrics.py with per-image predictions (D23) confirmed")

LABELS = ["1.0h", "2.0h", "3.0h", "4.0h"]
ROOTS  = {"1.0h": f"{CA}/curve_a", "2.0h": f"{CA}/curve_a", "3.0h": f"{CB}/curve_b", "4.0h": f"{CB}/curve_b"}
ARMS   = {"gan": "gan", "lora": "lora"}
for lab in LABELS:
    for arm in ARMS:
        for c in ["akiec", "bcc", "df", "mel", "vasc"]:
            n = len(glob.glob(f"{ROOTS[lab]}/curve_{lab}/{arm}/{c}/*.png"))
            assert n == 500, (lab, arm, c, n)
print("curve images: 4 checkpoints x 2 arms x 5 classes x 500 = 20,000 confirmed")

code: metrics.py with per-image predictions (D23) confirmed
curve images: 4 checkpoints x 2 arms x 5 classes x 500 = 20,000 confirmed


In [2]:
import sys, shutil, torch, pandas as pd
WORK = "/kaggle/working/repo"
os.makedirs(f"{WORK}/src", exist_ok=True); os.makedirs(f"{WORK}/config", exist_ok=True)
for f in os.listdir(CODE):
    shutil.copy(f"{CODE}/{f}", f"{WORK}/config/{f}" if f.endswith(".yaml") else f"{WORK}/src/{f}")
sys.path.insert(0, WORK); os.chdir(WORK)
print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
assert torch.__version__.startswith("2.11")

from src.run_registry import init_run
root = init_run("run1_cloud", root="/kaggle/working/results", git_hash="6c8939b")
from src.data import HAM10000, make_loader, build_image_index, assert_no_synthetic_leakage, AUG_CLASSES
assert_no_synthetic_leakage(f"{SPL}/val.csv", f"{SPL}/test.csv")
index = build_image_index(IMG)
NW = 4
val_ds  = HAM10000(f"{SPL}/val.csv",  IMG, size=128, augment=False, index=index)
test_ds = HAM10000(f"{SPL}/test.csv", IMG, size=128, augment=False, index=index)
val_loader  = make_loader(val_ds,  batch_size=32, shuffle=False, num_workers=NW)
test_loader = make_loader(test_ds, batch_size=32, shuffle=False, num_workers=NW)
print(f"val {len(val_ds)}  test {len(test_ds)}")

2.11.0+cu128 | GPU: Tesla T4
[registry] run1_cloud (PRIMARY - Kaggle free-tier GPU, matches proposal Table 8.2)
[registry] segment 1 on Tesla T4 | torch 2.11.0+cu128 | cuda 12.8
[registry] writing to /kaggle/working/results/run1_cloud
[data] no synthetic leakage in 2 partition(s)
[data] indexed 10015 images under /kaggle/input/datasets/ashwanisingh997/ham10000-images-1229692
val 1532  test 1502


In [3]:
def curve_rows(lab, arm):
    rows = []
    for c in AUG_CLASSES:
        files = sorted(glob.glob(f"{ROOTS[lab]}/curve_{lab}/{arm}/{c}/*.png"))
        assert len(files) == 500, (lab, arm, c, len(files))
        rows += [{"image_id": os.path.basename(f)[:-4], "dx": c, "path": f, "synthetic": True} for f in files]
    return pd.DataFrame(rows)

EXTRA = {(lab, arm): curve_rows(lab, arm) for lab in LABELS for arm in ARMS}
print({k: len(v) for k, v in EXTRA.items()})

{('1.0h', 'gan'): 2500, ('1.0h', 'lora'): 2500, ('2.0h', 'gan'): 2500, ('2.0h', 'lora'): 2500, ('3.0h', 'gan'): 2500, ('3.0h', 'lora'): 2500, ('4.0h', 'gan'): 2500, ('4.0h', 'lora'): 2500}


In [4]:
import json, numpy as np
from src.utils import set_seed
from src.classifier import build_model, train_classifier
from src.metrics import evaluate, save_metrics

SMOKE  = False   # True: 1 epoch, seed 42, 1.0h only. False: the real 24 runs.
SEEDS  = [42] if SMOKE else [42, 43, 44]
LABS   = ["1.0h"] if SMOKE else LABELS
EPOCHS = 1 if SMOKE else 30
PRE    = "SMOKE_" if SMOKE else ""

for seed in SEEDS:              # seed outermost: after the first 8 runs a full seed-42 curve exists
    for lab in LABS:
        for arm in ARMS:
            name = f"{PRE}CURVE_{arm}_{lab}_seed{seed}"
            if os.path.exists(f"{root}/metrics/{name}.json"):
                print(f"skip {name}"); continue
            print(f"\n{'='*60}\n{name}\n{'='*60}")
            set_seed(seed)
            tr = HAM10000(f"{SPL}/train.csv", IMG, size=128, augment=False, index=index, extra_rows=EXTRA[(lab, arm)])
            assert len(tr) == 6981 + 2500, len(tr)
            tr_ld = make_loader(tr, batch_size=32, shuffle=True, num_workers=NW)
            m, _ = train_classifier(build_model(), tr_ld, val_loader, epochs=EPOCHS, lr=1e-4,
                                    weight_decay=1e-4, patience=7, tag=name, log_dir=f"{root}/logs")
            save_metrics(evaluate(m, test_loader, bootstrap=True, seed=seed), name, root=str(root))
            del m; torch.cuda.empty_cache()
print("ALL CURVE RUNS COMPLETE")


CURVE_gan_1.0h_seed42
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 185MB/s]


ep 00  loss 0.7684  val macroF1 0.5157  val acc 0.7768
ep 01  loss 0.3283  val macroF1 0.5586  val acc 0.7879
ep 02  loss 0.1627  val macroF1 0.5803  val acc 0.7931
ep 03  loss 0.0819  val macroF1 0.5617  val acc 0.7937
ep 04  loss 0.0537  val macroF1 0.5878  val acc 0.7846
ep 05  loss 0.0460  val macroF1 0.6233  val acc 0.8198
ep 06  loss 0.0378  val macroF1 0.5987  val acc 0.8035
ep 07  loss 0.0351  val macroF1 0.5773  val acc 0.7905
ep 08  loss 0.0357  val macroF1 0.6195  val acc 0.7983
ep 09  loss 0.0209  val macroF1 0.5737  val acc 0.8087
ep 10  loss 0.0206  val macroF1 0.6047  val acc 0.8087
ep 11  loss 0.0194  val macroF1 0.5886  val acc 0.8068
ep 12  loss 0.0099  val macroF1 0.6326  val acc 0.8146
ep 13  loss 0.0080  val macroF1 0.6159  val acc 0.8003
ep 14  loss 0.0055  val macroF1 0.5993  val acc 0.7996
ep 15  loss 0.0073  val macroF1 0.6171  val acc 0.7957
ep 16  loss 0.0058  val macroF1 0.6041  val acc 0.8055
ep 17  loss 0.0030  val macroF1 0.6192  val acc 0.8003
ep 18  los

In [5]:
rows = []
for lab in LABS:
    for arm in ARMS:
        rs = [json.load(open(f"{root}/metrics/{PRE}CURVE_{arm}_{lab}_seed{s}.json")) for s in SEEDS]
        for metric in ["macro_f1", "melanoma_recall", "balanced_accuracy", "macro_auroc"]:
            v = [r[metric] for r in rs]
            rows.append({"checkpoint": lab, "arm": arm, "metric": metric, "mean": np.mean(v),
                         "sd": np.std(v, ddof=1) if len(v) > 1 else float("nan"), "values": [round(x, 4) for x in v]})
summ = pd.DataFrame(rows).round(4)
summ.to_csv(f"{root}/metrics/{PRE}curve_summary.csv", index=False)
print(summ[summ.metric.isin(["macro_f1", "melanoma_recall"])].pivot_table(index="checkpoint", columns=["metric", "arm"], values="mean").to_string())
npz = glob.glob(f"{root}/metrics/{PRE}CURVE_*_predictions.npz")
assert len(npz) == len(SEEDS) * len(LABS) * 2, len(npz)
print(f"{len(npz)} prediction files")
shutil.make_archive("/kaggle/working/run1_curve_results", "zip", "/kaggle/working/results")
print("saved: /kaggle/working/run1_curve_results.zip")

metric     macro_f1         melanoma_recall        
arm             gan    lora             gan    lora
checkpoint                                         
1.0h         0.6465  0.6533          0.4830  0.4990
2.0h         0.6444  0.6300          0.4271  0.4471
3.0h         0.6559  0.6315          0.3992  0.4651
4.0h         0.6234  0.6587          0.4691  0.4531
24 prediction files
saved: /kaggle/working/run1_curve_results.zip
